In [2]:
import pandas as pd
import utils
import plotly.graph_objects as go
import instrument

In [39]:
pair = "USD_CAD"
granularity = "H1"
ma_list = [16, 64] # Looking at 16 MA crossing 64 MA for trade entry
i_pair = instrument.Instrument.get_instrument_by_name(pair)

In [40]:
# Reading in EUR_USD's historical data, from pickle to df, using utils module
df = pd.read_pickle(utils.get_hist_data_filename(pair, granularity))
non_price_cols = ['time', 'volume']
price_cols = [x for x in df.columns if x not in non_price_cols] # Using list comprehension to isolate the price columns
df[price_cols] = df[price_cols].apply(pd.to_numeric)

In [41]:
# Creating df for moving average strategy using mid prices
df_ma = df[['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c']].copy()

In [42]:
df_ma.head(10)

,time,volume,mid_o,mid_h,mid_l,mid_c
0,2025-09-25T22:00:00.000000000Z,688,1.39385,1.39418,1.39372,1.39414
1,2025-09-25T23:00:00.000000000Z,1688,1.39412,1.39452,1.39400,1.39416
2,2025-09-26T00:00:00.000000000Z,4281,1.39416,1.39458,1.39379,1.39390
3,2025-09-26T01:00:00.000000000Z,3923,1.39391,1.39449,1.39369,1.39437
4,2025-09-26T02:00:00.000000000Z,3243,1.39436,1.39440,1.39367,1.39394
5,2025-09-26T03:00:00.000000000Z,2356,1.39394,1.39404,1.39366,1.39370
6,2025-09-26T04:00:00.000000000Z,2662,1.39370,1.39390,1.39344,1.39355
7,2025-09-26T05:00:00.000000000Z,2290,1.39356,1.39408,1.39352,1.39406
8,2025-09-26T06:00:00.000000000Z,4350,1.39405,1.39478,1.39390,1.39427
9,2025-09-26T07:00:00.000000000Z,4869,1.39428,1.39482,1.39394,1.39458


In [43]:
# Looping through MA list, calculating and adding each MA to the DF
for ma in ma_list:
    df_ma[f'MA_{ma}'] = df_ma['mid_c'].rolling(window=ma).mean()

In [44]:
df_ma.dropna(inplace=True) # OR df_ma = df_ma.dropna()

In [45]:
# Dropna retains original index, we need to reset index to easily reference in future
df_ma.reset_index(drop=True, inplace=True) # Modify existing DF by dropping the existing index and resetting it

In [46]:
# Calculating the difference in MAs in order to find crossing points
df_ma['DIFF'] = df_ma['MA_16'] - df_ma['MA_64']
df_ma['DIFF_prev'] = df_ma['DIFF'].shift(1) # similar to LAG() window function (-1 for LEAD())

In [47]:
# Trade signal is when diff and diff_prev have opposite signs => They are crossing/ touching

def is_trade(row):
    if row['DIFF'] >= 0 and row['DIFF_prev'] < 0:
        return 1
    if row['DIFF'] <= 0 and row['DIFF_prev'] > 0:
        return -1
    else:
        return 0

In [48]:
# apply the is_trade() function while passing each row as the default param
# axis = 1 means we are passsing in rows
# axis = 0 means we are passing in the columns
df_ma['IS_TRADE'] = df_ma.apply(is_trade, axis=1) 

In [49]:
df_ma.head()

,time,volume,mid_o,mid_h,mid_l,mid_c,MA_16,MA_64,DIFF,DIFF_prev,IS_TRADE
0,2025-09-30T13:00:00.000000000Z,7202,1.39191,1.39298,1.39073,1.39146,1.391653,1.392839,-0.001186,NaN,0
1,2025-09-30T14:00:00.000000000Z,10048,1.39145,1.39157,1.38966,1.39145,1.391654,1.392797,-0.001143,-0.001186,0
2,2025-09-30T15:00:00.000000000Z,7047,1.39147,1.39347,1.39134,1.39295,1.391726,1.392778,-0.001052,-0.001143,0
3,2025-09-30T16:00:00.000000000Z,4935,1.39294,1.39362,1.39192,1.39216,1.391726,1.392751,-0.001025,-0.001052,0
4,2025-09-30T17:00:00.000000000Z,3829,1.39215,1.39249,1.39161,1.39184,1.391711,1.392711,-0.001001,-0.001025,0


In [50]:
df_trades = df_ma[df_ma['IS_TRADE'] != 0].copy()

In [51]:
# Calculating diff between next - current mid_c price
# Using pipLocation from instrument class to convert diff into pips
df_trades['DELTA'] = (df_trades['mid_c'].diff() / i_pair.pipLocation).shift(-1)
df_trades['GAIN'] = df_trades["DELTA"] * df_trades['IS_TRADE']

In [52]:
df_trades.head()

,time,volume,mid_o,mid_h,mid_l,mid_c,MA_16,MA_64,DIFF,DIFF_prev,IS_TRADE,DELTA,GAIN
21,2025-10-01T10:00:00.000000000Z,4873,1.39274,1.39370,1.39264,1.39345,1.392221,1.392164,0.000057,-0.000053,1,12.9,12.9
89,2025-10-06T06:00:00.000000000Z,5407,1.39534,1.39562,1.39472,1.39474,1.395361,1.395361,0.000000,0.000029,-1,6.4,-6.4
97,2025-10-06T14:00:00.000000000Z,7140,1.39637,1.39653,1.39530,1.39538,1.395507,1.395540,-0.000033,0.000066,-1,11.2,-11.2
133,2025-10-08T02:00:00.000000000Z,3889,1.39647,1.39686,1.39621,1.39650,1.395353,1.395300,0.000054,-0.000030,1,-13.6,-13.6
152,2025-10-08T21:00:00.000000000Z,437,1.39518,1.39526,1.39504,1.39514,1.395301,1.395315,-0.000014,0.000090,-1,59.8,-59.8


In [53]:
df_trades['GAIN'].sum()

np.float64(570.1000000000222)

In [54]:
# df_plot = df_ma.tail(100)
df_plot = df_ma.iloc[140:170].copy()

In [55]:
fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df_plot['time'], open=df_plot['mid_o'], high=df_plot['mid_h'], low=df_plot['mid_l'], close=df_plot['mid_c'],
    line=dict(width=1), opacity=1,
    increasing_fillcolor= '#24A06B',
    decreasing_fillcolor= '#CC2E3C',
    increasing_line_color= '#2EC886',
    decreasing_line_color= '#FF3A4C' 
))
for ma in ma_list:
    col = f"MA_{ma}"
    fig.add_trace(go.Scatter(x = df_plot['time'],
                            y = df_plot[col],
                            line = dict(width = 2),
                            line_shape = 'spline',
                            name = col))

fig.update_layout(width=1450, height=600,
    margin=dict(l=10, r=10, t=10, b=10),
    font=dict(size=10, color='#E1E1E1'),
    paper_bgcolor= '#1E1E1E',
    plot_bgcolor= '#1E1E1E'
)

fig.update_xaxes(
    gridcolor='#1F292F',
    showgrid=True, fixedrange=True, rangeslider=dict(visible=False)
)

fig.update_yaxes(
    gridcolor='#1F292F',
    showgrid=True
)

fig.show()